# 06 — Transfer canonical historical scVI/Leiden results to the full-gene object

This notebook is the **branch bridge** between the reproducibility rerun
(`04`–`05`) and the article-defining downstream analysis (`07+`).

The fresh scVI rerun is retained as a reproducibility audit. The article branch,
however, must use the exact cluster assignments generated by the archived
executed analysis when the fresh stochastic rerun does not reproduce the
historical partition exactly.

The archived article-defining checkpoint is:

- 351,427 biological cells;
- 8,000 HVGs;
- `X_scVI`: `(351427, 30)`;
- `X_umap`: `(351427, 2)`;
- Leiden main key: `leiden_scvi_HVG8000_r0_6`;
- `r = 0.60` → **18 clusters**;
- cluster-size minimum `105`, maximum `61,646`, median `14,336.5`.

Historical dataset nomenclature is normalized only for cell identity matching:

- `DS5_GSE182109` → `DS3_GSE182109`
- `DS13_GSE278450` → `DS4_GSE278450`

The canonical historical **cluster assignments and embeddings are never
recomputed or relabeled**.

## Memory model

The 15,176-gene count matrix is never loaded into RAM. The notebook:

1. reads only metadata and historical embeddings;
2. builds/reuses a lightweight canonical historical reference;
3. sequentially copies the current full-gene count H5AD;
4. patches only `/obs`, `/obsm`, and provenance in `/uns`;
5. validates the final H5AD in backed mode.

This is intended for the current Windows / 16-GB-RAM workflow.

## One-time canonical source configuration

The private historical source path is **not hard-coded in the repository**.

On the first run, set the environment variable:

```text
GLIOMA_CANONICAL_SCVI_H5AD
```

to the archived executed H5AD that contains the historical `X_scVI`, `X_umap`
and resolution-sweep Leiden columns.

After this notebook creates the lightweight canonical reference under
`data/processed/integrated/`, future runs can reuse that generated reference
without needing the external historical H5AD.

In [1]:
# =========================
# 0) IMPORTS AND CONFIG
# =========================

import os
import gc
import re
import json
import shutil
import warnings
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

try:
    from anndata.io import read_elem, write_elem
except ImportError:
    from anndata._io.specs import read_elem, write_elem

warnings.filterwarnings("ignore")


def _find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
        ):
            return candidate

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        _find_repository_root(),
    )
).expanduser().resolve()

INTEGRATED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "integrated"
)

METADATA_DIR = (
    PROJECT_DIR
    / "metadata"
)

for directory in (
    INTEGRATED_DIR,
    METADATA_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------------
# Current reproducible pipeline inputs
# ------------------------------------------------------------------

# Large 15,176-gene count object from notebook 01.
FULLGENE_COUNTS_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_postQC_merged_counts.h5ad"
)

# Lightweight standardized metadata from notebook 02.
STANDARDIZED_METADATA_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_postQC_merged_counts_metadata.h5ad"
)


# ------------------------------------------------------------------
# Canonical historical source / generated lightweight reference
# ------------------------------------------------------------------

CANONICAL_REFERENCE_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_HVG8000_scVI_canonical_historical_reference.h5ad"
)

_historical_source_env = os.environ.get(
    "GLIOMA_CANONICAL_SCVI_H5AD"
)

HISTORICAL_SOURCE_H5AD = (
    Path(_historical_source_env)
    .expanduser()
    .resolve()
    if _historical_source_env
    else None
)


# ------------------------------------------------------------------
# Output
# ------------------------------------------------------------------

# Generic filename is the single physical downstream backbone.
FULLGENE_OUTPUT_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters.h5ad"
)

# Optional filesystem hard-link alias. No second 16+ GiB copy is created.
FULLGENE_R06_ALIAS_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters_r0_6.h5ad"
)


# ------------------------------------------------------------------
# Historical article checkpoint
# ------------------------------------------------------------------

MAIN_CLUSTER_KEY = (
    "leiden_scvi_HVG8000_r0_6"
)

REFINEMENT_CLUSTER_KEY = (
    "leiden_scvi_HVG8000_r0_7"
)

EXPECTED_N_CELLS = 351_427
EXPECTED_HVG_GENES = 8_000
EXPECTED_FULLGENE_GENES = 15_176

EXPECTED_X_SCVI_SHAPE = (
    351_427,
    30,
)

EXPECTED_X_UMAP_SHAPE = (
    351_427,
    2,
)

HISTORICAL_R06_N_CLUSTERS = 18
HISTORICAL_R06_MIN_CLUSTER_SIZE = 105
HISTORICAL_R06_MAX_CLUSTER_SIZE = 61_646
HISTORICAL_R06_MEDIAN_CLUSTER_SIZE = 14_336.5


# ------------------------------------------------------------------
# Reports
# ------------------------------------------------------------------

TRANSFER_REPORT_TSV = (
    METADATA_DIR
    / "transfer_canonical_HVG8000_scVI_to_fullgene_report.tsv"
)

TRANSFERRED_OBS_COLUMNS_TSV = (
    METADATA_DIR
    / "transfer_canonical_HVG8000_scVI_obs_columns.tsv"
)

MAIN_CLUSTER_SIZES_TSV = (
    METADATA_DIR
    / "fullgene_canonical_HVG8000_r0_6_cluster_sizes.tsv"
)

MAIN_CLUSTER_COMPOSITION_TSV = (
    METADATA_DIR
    / "fullgene_canonical_HVG8000_r0_6_cluster_composition.tsv"
)

CLUSTER_KEY_SUMMARY_TSV = (
    METADATA_DIR
    / "fullgene_canonical_HVG8000_cluster_key_summary.tsv"
)

TRANSFER_MANIFEST_TSV = (
    METADATA_DIR
    / "transfer_canonical_HVG8000_scVI_to_fullgene_manifest.tsv"
)


def write_tsv_replace(
    dataframe,
    path,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if path.exists():
        path.unlink()

    dataframe.to_csv(
        path,
        sep="\t",
        index=False,
    )

    print(
        f"[SAVED] {path} "
        f"shape={dataframe.shape}"
    )


def replace_h5_key(
    parent,
    key,
    value,
):
    if key in parent:
        del parent[key]

    write_elem(
        parent,
        key,
        value,
    )


def stored_shape(
    node,
):
    if isinstance(
        node,
        h5py.Dataset,
    ):
        return tuple(
            int(x)
            for x in node.shape
        )

    shape = node.attrs.get(
        "shape"
    )

    if shape is None:
        raise KeyError(
            f"{node.name} has no stored shape."
        )

    return tuple(
        int(x)
        for x in shape
    )


print(
    "PROJECT_DIR:",
    PROJECT_DIR,
)

print(
    "FULLGENE_COUNTS_H5AD:",
    FULLGENE_COUNTS_H5AD,
    "| exists:",
    FULLGENE_COUNTS_H5AD.exists(),
)

print(
    "STANDARDIZED_METADATA_H5AD:",
    STANDARDIZED_METADATA_H5AD,
    "| exists:",
    STANDARDIZED_METADATA_H5AD.exists(),
)

print(
    "CANONICAL_REFERENCE_H5AD:",
    CANONICAL_REFERENCE_H5AD,
    "| exists:",
    CANONICAL_REFERENCE_H5AD.exists(),
)

print(
    "Historical source configured:",
    HISTORICAL_SOURCE_H5AD is not None,
)

PROJECT_DIR: G:\Repository\glioma-cnv-single-cell-atlas
FULLGENE_COUNTS_H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_postQC_merged_counts.h5ad | exists: True
STANDARDIZED_METADATA_H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_postQC_merged_counts_metadata.h5ad | exists: True
CANONICAL_REFERENCE_H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_HVG8000_scVI_canonical_historical_reference.h5ad | exists: False
Historical source configured: False


In [2]:
# =========================
# 1) CELL-IDENTITY HELPERS
# =========================

def normalize_dataset_name(
    value,
):
    return (
        str(value)
        .replace(
            "DS5_GSE182109",
            "DS3_GSE182109",
        )
        .replace(
            "DS13_GSE278450",
            "DS4_GSE278450",
        )
    )


def normalize_obs_name(
    value,
):
    return (
        str(value)
        .replace(
            "DS5_GSE182109",
            "DS3_GSE182109",
        )
        .replace(
            "DS13_GSE278450",
            "DS4_GSE278450",
        )
    )


def normalize_sample_name(
    value,
):
    return re.sub(
        r"^GSM\d+_",
        "",
        str(value),
    )


def extract_nucleotide_barcode(
    value,
):
    candidates = re.findall(
        r"[ACGTN]{8,30}",
        str(value).upper(),
    )

    if not candidates:
        return None

    return max(
        candidates,
        key=len,
    )


def canonical_cell_keys(
    obs,
):
    """
    Dataset-specific biological cell identity established during audit.

    DS1 / DS3 / DS4:
        normalized obs_name is exact.

    DS2:
        normalized sample_id + nucleotide barcode is exact.
    """

    required = [
        "core_dataset",
        "sample_id",
        "original_cell_barcode",
    ]

    missing = [
        column
        for column in required
        if column not in obs.columns
    ]

    if missing:
        raise KeyError(
            "Cannot build canonical cell IDs; "
            f"missing columns: {missing}"
        )

    datasets = (
        obs["core_dataset"]
        .astype(str)
        .map(
            normalize_dataset_name
        )
    )

    keys = pd.Series(
        index=obs.index,
        dtype="object",
    )


    for dataset in (
        "DS1_GSE103224",
        "DS3_GSE182109",
        "DS4_GSE278450",
    ):
        mask = (
            datasets
            == dataset
        )

        keys.loc[mask] = [
            (
                dataset
                + "||"
                + normalize_obs_name(
                    obs_name
                )
            )
            for obs_name
            in obs.index[mask]
        ]


    ds2_mask = (
        datasets
        == "DS2_GSE141383"
    )

    ds2_samples = (
        obs.loc[
            ds2_mask,
            "sample_id",
        ]
        .astype(str)
        .map(
            normalize_sample_name
        )
    )

    ds2_barcodes = (
        obs.loc[
            ds2_mask,
            "original_cell_barcode",
        ]
        .astype(str)
        .map(
            extract_nucleotide_barcode
        )
    )

    if ds2_barcodes.isna().any():
        raise RuntimeError(
            "DS2 contains cells without an "
            "extractable nucleotide barcode."
        )

    keys.loc[ds2_mask] = (
        "DS2_GSE141383||"
        + ds2_samples
        + "||"
        + ds2_barcodes
    )


    if keys.isna().any():
        unresolved = int(
            keys.isna().sum()
        )

        raise RuntimeError(
            f"{unresolved} cells did not receive "
            "a canonical biological ID."
        )


    if keys.nunique() != len(keys):
        raise RuntimeError(
            "Canonical biological cell IDs "
            "are not unique."
        )


    return keys


def align_source_to_target(
    source_obs,
    target_obs,
):
    source_keys = canonical_cell_keys(
        source_obs
    )

    target_keys = canonical_cell_keys(
        target_obs
    )

    source_index = pd.Index(
        source_keys.to_numpy()
    )

    target_index = pd.Index(
        target_keys.to_numpy()
    )

    same_set = (
        len(source_index)
        == len(target_index)
        and source_index.isin(
            target_index
        ).all()
        and target_index.isin(
            source_index
        ).all()
    )

    if not same_set:
        raise RuntimeError(
            "Canonical biological cell sets "
            "do not match."
        )

    position = pd.Series(
        np.arange(
            len(source_index),
            dtype=np.int64,
        ),
        index=source_index,
    )

    indexer = (
        position
        .loc[target_index]
        .to_numpy()
    )

    return indexer

In [3]:
# =========================
# 2) READ CURRENT METADATA + VALIDATE FULL-GENE COUNT BACKBONE
# =========================

if not FULLGENE_COUNTS_H5AD.exists():
    raise FileNotFoundError(
        FULLGENE_COUNTS_H5AD
    )

if not STANDARDIZED_METADATA_H5AD.exists():
    raise FileNotFoundError(
        STANDARDIZED_METADATA_H5AD
    )


with h5py.File(
    FULLGENE_COUNTS_H5AD,
    "r",
) as handle:

    if "obs" not in handle:
        raise KeyError(
            "Full-gene count H5AD is missing /obs."
        )

    counts_obs = read_elem(
        handle["obs"]
    )

    x_shape = stored_shape(
        handle["X"]
    )

    if "layers" not in handle:
        raise KeyError(
            "Full-gene count H5AD is missing /layers."
        )

    if "counts" not in handle["layers"]:
        raise KeyError(
            "Full-gene count H5AD is missing layers['counts']."
        )

    counts_layer_shape = stored_shape(
        handle["layers"]["counts"]
    )


with h5py.File(
    STANDARDIZED_METADATA_H5AD,
    "r",
) as handle:

    standardized_obs = read_elem(
        handle["obs"]
    )


print(
    "Full-gene X shape:",
    x_shape,
)

print(
    "Full-gene counts-layer shape:",
    counts_layer_shape,
)

print(
    "Count-object obs rows:",
    len(counts_obs),
)

print(
    "Standardized metadata rows:",
    len(standardized_obs),
)


if x_shape != (
    EXPECTED_N_CELLS,
    EXPECTED_FULLGENE_GENES,
):
    raise AssertionError(
        "Unexpected full-gene X shape: "
        f"{x_shape}"
    )

if counts_layer_shape != x_shape:
    raise AssertionError(
        "X and layers['counts'] shapes differ."
    )

if len(counts_obs) != EXPECTED_N_CELLS:
    raise AssertionError(
        "Unexpected full-gene cell count."
    )

if len(standardized_obs) != EXPECTED_N_CELLS:
    raise AssertionError(
        "Unexpected standardized metadata cell count."
    )


# Align standardized metadata to the physical row order of the count matrix.
metadata_indexer = align_source_to_target(
    standardized_obs,
    counts_obs,
)

standardized_obs_aligned = (
    standardized_obs.iloc[
        metadata_indexer
    ].copy()
)

standardized_obs_aligned.index = (
    counts_obs.index.copy()
)


print(
    "Standardized metadata aligned to "
    "the full-gene count row order."
)

print(
    "Final current nomenclature levels:",
    sorted(
        standardized_obs_aligned[
            "core_dataset"
        ]
        .astype(str)
        .unique()
        .tolist()
    ),
)

Full-gene X shape: (351427, 15176)
Full-gene counts-layer shape: (351427, 15176)
Count-object obs rows: 351427
Standardized metadata rows: 351427
Standardized metadata aligned to the full-gene count row order.
Final current nomenclature levels: ['DS1_GSE103224', 'DS2_GSE141383', 'DS3_GSE182109', 'DS4_GSE278450']


In [5]:
# =========================
# ONE-TIME LOCAL HISTORICAL SOURCE
# =========================

import os
from pathlib import Path

os.environ["GLIOMA_CANONICAL_SCVI_H5AD"] = (
    r"G:\DATA\GBM_scRNA_project\processed\integrated"
    r"\GBM_core_4datasets_HVG8000_scVI_clustering_resolution_QC_lowmem.h5ad"
)

HISTORICAL_SOURCE_H5AD = Path(
    os.environ["GLIOMA_CANONICAL_SCVI_H5AD"]
).expanduser().resolve()

print("Historical source:")
print(HISTORICAL_SOURCE_H5AD)
print("Exists:", HISTORICAL_SOURCE_H5AD.exists())

if not HISTORICAL_SOURCE_H5AD.exists():
    raise FileNotFoundError(HISTORICAL_SOURCE_H5AD)

Historical source:
G:\DATA\GBM_scRNA_project\processed\integrated\GBM_core_4datasets_HVG8000_scVI_clustering_resolution_QC_lowmem.h5ad
Exists: True


In [6]:
# =========================
# 3) BUILD OR REUSE LIGHTWEIGHT CANONICAL HISTORICAL REFERENCE
# =========================

def validate_canonical_reference(
    path,
):
    if not Path(path).exists():
        return False

    check = sc.read_h5ad(
        path,
        backed="r",
    )

    try:
        if check.shape != (
            EXPECTED_N_CELLS,
            EXPECTED_HVG_GENES,
        ):
            return False

        if MAIN_CLUSTER_KEY not in check.obs.columns:
            return False

        if "X_scVI" not in check.obsm:
            return False

        if "X_umap" not in check.obsm:
            return False

        if tuple(
            check.obsm["X_scVI"].shape
        ) != EXPECTED_X_SCVI_SHAPE:
            return False

        if tuple(
            check.obsm["X_umap"].shape
        ) != EXPECTED_X_UMAP_SHAPE:
            return False

        counts = (
            check.obs[
                MAIN_CLUSTER_KEY
            ]
            .astype(str)
            .value_counts()
        )

        if counts.shape[0] != HISTORICAL_R06_N_CLUSTERS:
            return False

        if int(counts.min()) != HISTORICAL_R06_MIN_CLUSTER_SIZE:
            return False

        if int(counts.max()) != HISTORICAL_R06_MAX_CLUSTER_SIZE:
            return False

        if not np.isclose(
            float(counts.median()),
            HISTORICAL_R06_MEDIAN_CLUSTER_SIZE,
        ):
            return False

        return True

    finally:
        check.file.close()


if validate_canonical_reference(
    CANONICAL_REFERENCE_H5AD
):
    print(
        "[REUSING] validated lightweight "
        "canonical historical reference:"
    )

    print(
        CANONICAL_REFERENCE_H5AD
    )

else:
    if HISTORICAL_SOURCE_H5AD is None:
        raise FileNotFoundError(
            "No validated canonical reference exists yet, and "
            "GLIOMA_CANONICAL_SCVI_H5AD is not set. "
            "Set that environment variable to the archived "
            "executed historical clustering H5AD, restart the "
            "kernel, and run this notebook again."
        )

    if not HISTORICAL_SOURCE_H5AD.exists():
        raise FileNotFoundError(
            HISTORICAL_SOURCE_H5AD
        )

    print(
        "Building lightweight canonical reference from:"
    )

    print(
        HISTORICAL_SOURCE_H5AD
    )


    with h5py.File(
        HISTORICAL_SOURCE_H5AD,
        "r",
    ) as handle:

        historical_obs = read_elem(
            handle["obs"]
        )

        historical_var = read_elem(
            handle["var"]
        )

        if (
            "obsm" not in handle
            or "X_scVI" not in handle["obsm"]
            or "X_umap" not in handle["obsm"]
        ):
            raise KeyError(
                "Historical source is missing "
                "X_scVI and/or X_umap."
            )

        historical_x_scvi = np.asarray(
            handle[
                "obsm"
            ][
                "X_scVI"
            ],
            dtype=np.float32,
        )

        historical_x_umap = np.asarray(
            handle[
                "obsm"
            ][
                "X_umap"
            ],
            dtype=np.float32,
        )


    if len(historical_obs) != EXPECTED_N_CELLS:
        raise AssertionError(
            "Unexpected historical cell count."
        )

    if len(historical_var) != EXPECTED_HVG_GENES:
        raise AssertionError(
            "Unexpected historical HVG count."
        )

    if historical_x_scvi.shape != EXPECTED_X_SCVI_SHAPE:
        raise AssertionError(
            "Unexpected historical X_scVI shape."
        )

    if historical_x_umap.shape != EXPECTED_X_UMAP_SHAPE:
        raise AssertionError(
            "Unexpected historical X_umap shape."
        )

    if MAIN_CLUSTER_KEY not in historical_obs.columns:
        raise KeyError(
            f"Historical source is missing {MAIN_CLUSTER_KEY!r}."
        )


    historical_main_counts = (
        historical_obs[
            MAIN_CLUSTER_KEY
        ]
        .astype(str)
        .value_counts()
    )

    print(
        "Historical r=0.60 clusters:",
        historical_main_counts.shape[0],
    )

    print(
        "Historical min/max/median:",
        int(
            historical_main_counts.min()
        ),
        int(
            historical_main_counts.max()
        ),
        float(
            historical_main_counts.median()
        ),
    )


    if historical_main_counts.shape[0] != HISTORICAL_R06_N_CLUSTERS:
        raise AssertionError(
            "Historical source does not reproduce "
            "the archived 18-cluster checkpoint."
        )

    if int(
        historical_main_counts.min()
    ) != HISTORICAL_R06_MIN_CLUSTER_SIZE:
        raise AssertionError(
            "Historical minimum cluster size changed."
        )

    if int(
        historical_main_counts.max()
    ) != HISTORICAL_R06_MAX_CLUSTER_SIZE:
        raise AssertionError(
            "Historical maximum cluster size changed."
        )

    if not np.isclose(
        float(
            historical_main_counts.median()
        ),
        HISTORICAL_R06_MEDIAN_CLUSTER_SIZE,
    ):
        raise AssertionError(
            "Historical median cluster size changed."
        )


    # Align historical information to the CURRENT standardized cell order.
    historical_indexer = align_source_to_target(
        historical_obs,
        standardized_obs_aligned,
    )

    historical_obs_aligned = (
        historical_obs.iloc[
            historical_indexer
        ].copy()
    )

    historical_x_scvi_aligned = (
        historical_x_scvi[
            historical_indexer
        ]
    )

    historical_x_umap_aligned = (
        historical_x_umap[
            historical_indexer
        ]
    )


    # Preserve current standardized metadata; transfer only historical
    # Leiden assignments so old DS5/DS13 and old DS2 naming never leak
    # back into the current pipeline.
    canonical_obs = (
        standardized_obs_aligned.copy()
    )

    historical_leiden_columns = [
        column
        for column
        in historical_obs_aligned.columns
        if column.startswith(
            "leiden_scvi_"
        )
    ]

    for column in historical_leiden_columns:
        canonical_obs[column] = (
            historical_obs_aligned[
                column
            ]
            .astype(str)
            .astype("category")
            .to_numpy()
        )


    canonical_obs[
        "leiden_scvi_main"
    ] = (
        canonical_obs[
            MAIN_CLUSTER_KEY
        ]
        .astype(str)
        .astype("category")
    )

    canonical_obs[
        "leiden_scvi_HVG8000_main"
    ] = (
        canonical_obs[
            MAIN_CLUSTER_KEY
        ]
        .astype(str)
        .astype("category")
    )


    if REFINEMENT_CLUSTER_KEY in canonical_obs.columns:
        canonical_obs[
            "leiden_scvi_HVG8000_refinement"
        ] = (
            canonical_obs[
                REFINEMENT_CLUSTER_KEY
            ]
            .astype(str)
            .astype("category")
        )


    canonical_reference = ad.AnnData(
        X=None,
        obs=canonical_obs,
        var=historical_var.copy(),
        obsm={
            "X_scVI":
                historical_x_scvi_aligned,

            "X_umap":
                historical_x_umap_aligned,
        },
    )


    canonical_reference.uns[
        "canonical_historical_scvi_reference"
    ] = {
        "description": (
            "Article-defining historical scVI/Leiden artifact "
            "aligned to current standardized biological cell IDs."
        ),
        "main_cluster_key":
            MAIN_CLUSTER_KEY,
        "historical_r0_6_n_clusters":
            HISTORICAL_R06_N_CLUSTERS,
        "historical_dataset_rename":
            "DS5->DS3; DS13->DS4",
        "historical_source_path_was_external":
            True,
        "raw_expression_matrix_in_reference":
            False,
    }


    temporary_reference = (
        CANONICAL_REFERENCE_H5AD
        .with_suffix(
            ".tmp.h5ad"
        )
    )

    if temporary_reference.exists():
        temporary_reference.unlink()

    canonical_reference.write_h5ad(
        temporary_reference,
        compression="lzf",
    )

    os.replace(
        temporary_reference,
        CANONICAL_REFERENCE_H5AD,
    )

    print(
        "[SAVED canonical reference]",
        CANONICAL_REFERENCE_H5AD,
    )


    del historical_obs
    del historical_x_scvi
    del historical_x_umap
    del historical_obs_aligned
    del historical_x_scvi_aligned
    del historical_x_umap_aligned
    del canonical_reference

    gc.collect()


if not validate_canonical_reference(
    CANONICAL_REFERENCE_H5AD
):
    raise RuntimeError(
        "Canonical historical reference failed validation."
    )


print(
    "[OK] Canonical historical reference validated."
)

Building lightweight canonical reference from:
G:\DATA\GBM_scRNA_project\processed\integrated\GBM_core_4datasets_HVG8000_scVI_clustering_resolution_QC_lowmem.h5ad
Historical r=0.60 clusters: 18
Historical min/max/median: 105 61646 14336.5
[SAVED canonical reference] G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_HVG8000_scVI_canonical_historical_reference.h5ad
[OK] Canonical historical reference validated.


In [7]:
# =========================
# 4) READ CANONICAL REFERENCE FIELDS ONLY
# =========================

with h5py.File(
    CANONICAL_REFERENCE_H5AD,
    "r",
) as handle:

    canonical_obs = read_elem(
        handle["obs"]
    )

    canonical_var = read_elem(
        handle["var"]
    )

    canonical_x_scvi = np.asarray(
        handle["obsm"]["X_scVI"],
        dtype=np.float32,
    )

    canonical_x_umap = np.asarray(
        handle["obsm"]["X_umap"],
        dtype=np.float32,
    )


# The canonical reference was built in standardized/full-gene row order.
reference_indexer = align_source_to_target(
    canonical_obs,
    standardized_obs_aligned,
)

canonical_obs_aligned = (
    canonical_obs.iloc[
        reference_indexer
    ].copy()
)

canonical_x_scvi_aligned = (
    canonical_x_scvi[
        reference_indexer
    ]
)

canonical_x_umap_aligned = (
    canonical_x_umap[
        reference_indexer
    ]
)


leiden_cols = [
    column
    for column
    in canonical_obs_aligned.columns
    if column.startswith(
        "leiden_scvi_"
    )
]


if MAIN_CLUSTER_KEY not in leiden_cols:
    raise KeyError(
        f"{MAIN_CLUSTER_KEY!r} missing from canonical reference."
    )


print(
    "Canonical Leiden columns:",
    len(leiden_cols),
)

print(
    "Canonical X_scVI:",
    canonical_x_scvi_aligned.shape,
)

print(
    "Canonical X_umap:",
    canonical_x_umap_aligned.shape,
)

Canonical Leiden columns: 24
Canonical X_scVI: (351427, 30)
Canonical X_umap: (351427, 2)


In [8]:
# =========================
# 5) BUILD FINAL OBS TABLE
# =========================

final_obs = (
    standardized_obs_aligned.copy()
)


for column in leiden_cols:
    final_obs[column] = (
        canonical_obs_aligned[
            column
        ]
        .astype(str)
        .astype("category")
        .to_numpy()
    )


final_obs[
    "leiden_scvi_main"
] = (
    final_obs[
        MAIN_CLUSTER_KEY
    ]
    .astype(str)
    .astype("category")
)

final_obs[
    "leiden_scvi_HVG8000_main"
] = (
    final_obs[
        MAIN_CLUSTER_KEY
    ]
    .astype(str)
    .astype("category")
)


if REFINEMENT_CLUSTER_KEY in final_obs.columns:
    final_obs[
        "leiden_scvi_HVG8000_refinement"
    ] = (
        final_obs[
            REFINEMENT_CLUSTER_KEY
        ]
        .astype(str)
        .astype("category")
    )


main_counts = (
    final_obs[
        MAIN_CLUSTER_KEY
    ]
    .astype(str)
    .value_counts()
)


print(
    "Final main clusters:",
    main_counts.shape[0],
)

print(
    "Final min/max/median:",
    int(
        main_counts.min()
    ),
    int(
        main_counts.max()
    ),
    float(
        main_counts.median()
    ),
)


if main_counts.shape[0] != HISTORICAL_R06_N_CLUSTERS:
    raise AssertionError(
        "Canonical main clustering is not 18 clusters."
    )

if int(main_counts.min()) != HISTORICAL_R06_MIN_CLUSTER_SIZE:
    raise AssertionError(
        "Canonical minimum cluster size changed."
    )

if int(main_counts.max()) != HISTORICAL_R06_MAX_CLUSTER_SIZE:
    raise AssertionError(
        "Canonical maximum cluster size changed."
    )

if not np.isclose(
    float(
        main_counts.median()
    ),
    HISTORICAL_R06_MEDIAN_CLUSTER_SIZE,
):
    raise AssertionError(
        "Canonical median cluster size changed."
    )


transferred_columns = pd.DataFrame(
    {
        "obs_column":
            leiden_cols
            + [
                "leiden_scvi_main",
                "leiden_scvi_HVG8000_main",
                "leiden_scvi_HVG8000_refinement",
            ],

        "present_after_transfer":
            [
                (
                    column
                    in final_obs.columns
                )
                for column
                in (
                    leiden_cols
                    + [
                        "leiden_scvi_main",
                        "leiden_scvi_HVG8000_main",
                        "leiden_scvi_HVG8000_refinement",
                    ]
                )
            ],
    }
)


write_tsv_replace(
    transferred_columns,
    TRANSFERRED_OBS_COLUMNS_TSV,
)

display(
    transferred_columns
)

Final main clusters: 18
Final min/max/median: 105 61646 14336.5
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\transfer_canonical_HVG8000_scVI_obs_columns.tsv shape=(27, 2)


,obs_column,present_after_transfer
0,leiden_scvi_0.1,True
1,leiden_scvi_0.2,True
2,leiden_scvi_0.3,True
3,leiden_scvi_0.4,True
4,leiden_scvi_0.5,True
5,leiden_scvi_HVG8000_r0_1,True
6,leiden_scvi_HVG8000_r0_15,True
7,leiden_scvi_HVG8000_r0_2,True
8,leiden_scvi_HVG8000_r0_25,True
9,leiden_scvi_HVG8000_r0_3,True


In [9]:
# =========================
# 6) COPY FULL-GENE COUNT BACKBONE SEQUENTIALLY
# =========================

temporary_output = (
    FULLGENE_OUTPUT_H5AD
    .with_suffix(
        ".building.h5ad"
    )
)


if temporary_output.exists():
    temporary_output.unlink()


input_size = (
    FULLGENE_COUNTS_H5AD
    .stat()
    .st_size
)

free_space = shutil.disk_usage(
    FULLGENE_OUTPUT_H5AD.parent
).free


# Allow input-size plus a modest safety margin.
required_space = (
    input_size
    + 5 * 1024**3
)


print(
    "Full-gene input size GiB:",
    round(
        input_size / 1024**3,
        2,
    ),
)

print(
    "Free space GiB:",
    round(
        free_space / 1024**3,
        2,
    ),
)


if free_space < required_space:
    raise OSError(
        "Not enough free space to publish the canonical "
        "full-gene H5AD safely."
    )


print(
    "[COPY] full-gene count backbone:"
)

print(
    FULLGENE_COUNTS_H5AD
)

print(
    "->"
)

print(
    temporary_output
)


shutil.copyfile(
    FULLGENE_COUNTS_H5AD,
    temporary_output,
)


if (
    temporary_output.stat().st_size
    < input_size
):
    raise AssertionError(
        "Temporary output is unexpectedly smaller "
        "than the count backbone."
    )


print(
    "[COPY COMPLETE]"
)

Full-gene input size GiB: 11.11
Free space GiB: 774.19
[COPY] full-gene count backbone:
G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_postQC_merged_counts.h5ad
->
G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters.building.h5ad
[COPY COMPLETE]


In [10]:
# =========================
# 7) PATCH OBS / OBSM / PROVENANCE WITHOUT LOADING X
# =========================

transfer_info = {
    "description": (
        "Canonical article-defining historical scVI latent, UMAP, "
        "and Leiden assignments transferred onto the current "
        "15,176-gene full-gene count backbone."
    ),
    "canonical_reference_h5ad":
        str(
            CANONICAL_REFERENCE_H5AD
        ),
    "source_fullgene_counts_h5ad":
        str(
            FULLGENE_COUNTS_H5AD
        ),
    "source_standardized_metadata_h5ad":
        str(
            STANDARDIZED_METADATA_H5AD
        ),
    "main_cluster_key":
        MAIN_CLUSTER_KEY,
    "main_cluster_aliases": [
        "leiden_scvi_main",
        "leiden_scvi_HVG8000_main",
    ],
    "refinement_cluster_key":
        REFINEMENT_CLUSTER_KEY,
    "historical_r0_6_n_clusters":
        HISTORICAL_R06_N_CLUSTERS,
    "historical_dataset_nomenclature_mapping":
        "DS5_GSE182109->DS3_GSE182109; "
        "DS13_GSE278450->DS4_GSE278450",
    "current_standardized_metadata_preserved":
        True,
    "fresh_scvi_rerun_used_for_article_clusters":
        False,
    "graph_matrices_transferred":
        False,
    "full_expression_matrix_loaded_into_ram":
        False,
}


with h5py.File(
    temporary_output,
    "r+",
) as handle:

    replace_h5_key(
        handle,
        "obs",
        final_obs,
    )


    if "obsm" not in handle:
        write_elem(
            handle,
            "obsm",
            {},
        )


    replace_h5_key(
        handle["obsm"],
        "X_scVI",
        canonical_x_scvi_aligned,
    )

    replace_h5_key(
        handle["obsm"],
        "X_umap",
        canonical_x_umap_aligned,
    )


    if "uns" not in handle:
        write_elem(
            handle,
            "uns",
            {},
        )


    replace_h5_key(
        handle["uns"],
        "HVG8000_scVI_cluster_transfer_info",
        transfer_info,
    )


    handle.flush()


print(
    "[PATCH COMPLETE]"
)

[PATCH COMPLETE]


In [11]:
# =========================
# 8) VALIDATE TEMPORARY FINAL H5AD IN BACKED MODE
# =========================

check = sc.read_h5ad(
    temporary_output,
    backed="r",
)

try:
    print(
        "Temporary final object:",
        check,
    )

    print(
        "shape:",
        check.shape,
    )

    print(
        "layers:",
        list(
            check.layers.keys()
        ),
    )

    print(
        "obsm:",
        list(
            check.obsm.keys()
        ),
    )


    if check.shape != (
        EXPECTED_N_CELLS,
        EXPECTED_FULLGENE_GENES,
    ):
        raise AssertionError(
            "Final shape changed."
        )


    if "counts" not in check.layers:
        raise AssertionError(
            "Final object lost layers['counts']."
        )


    if MAIN_CLUSTER_KEY not in check.obs.columns:
        raise AssertionError(
            "Final object is missing the canonical main cluster."
        )


    if "X_scVI" not in check.obsm:
        raise AssertionError(
            "Final object is missing X_scVI."
        )


    if "X_umap" not in check.obsm:
        raise AssertionError(
            "Final object is missing X_umap."
        )


    if tuple(
        check.obsm[
            "X_scVI"
        ].shape
    ) != EXPECTED_X_SCVI_SHAPE:
        raise AssertionError(
            "Final X_scVI shape changed."
        )


    if tuple(
        check.obsm[
            "X_umap"
        ].shape
    ) != EXPECTED_X_UMAP_SHAPE:
        raise AssertionError(
            "Final X_umap shape changed."
        )


    validated_counts = (
        check.obs[
            MAIN_CLUSTER_KEY
        ]
        .astype(str)
        .value_counts()
    )


    if validated_counts.shape[0] != HISTORICAL_R06_N_CLUSTERS:
        raise AssertionError(
            "Final main clustering is not 18 clusters."
        )


    if int(
        validated_counts.min()
    ) != HISTORICAL_R06_MIN_CLUSTER_SIZE:
        raise AssertionError(
            "Final minimum cluster size changed."
        )


    if int(
        validated_counts.max()
    ) != HISTORICAL_R06_MAX_CLUSTER_SIZE:
        raise AssertionError(
            "Final maximum cluster size changed."
        )


    if not np.isclose(
        float(
            validated_counts.median()
        ),
        HISTORICAL_R06_MEDIAN_CLUSTER_SIZE,
    ):
        raise AssertionError(
            "Final median cluster size changed."
        )


    expected_dataset_levels = {
        "DS1_GSE103224",
        "DS2_GSE141383",
        "DS3_GSE182109",
        "DS4_GSE278450",
    }


    observed_dataset_levels = set(
        check.obs[
            "core_dataset"
        ]
        .astype(str)
        .unique()
    )


    if observed_dataset_levels != expected_dataset_levels:
        raise AssertionError(
            "Final core_dataset nomenclature is not "
            "the standardized DS1/DS2/DS3/DS4 scheme."
        )


    print(
        "\n[VALIDATED]"
    )

    print(
        "Canonical r=0.60 clusters:",
        validated_counts.shape[0],
    )

    print(
        "Canonical cluster min/max/median:",
        int(
            validated_counts.min()
        ),
        int(
            validated_counts.max()
        ),
        float(
            validated_counts.median()
        ),
    )

finally:
    check.file.close()

Temporary final object: AnnData object with n_obs × n_vars = 351427 × 15176 backed at 'G:\\Repository\\glioma-cnv-single-cell-atlas\\data\\processed\\integrated\\GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters.building.h5ad'
    obs: 'sample_id', 'geo_id', 'core_dataset', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', 'dataset_id', 'study_id', 'source_dataset', 'source_file', 'concat_key', 'patient_id', 'condition', 'batch_dataset', 'batch_sample', 'batch_patient', 'original_cell_barcode', 'diagnosis', 'disease_stage', 'tumor_status', 'platform', 'technology', 'study_batch', 'sample_batch', 'scvi_batch', 'analysis_group_major', 'analysis_group_detail', 'ready_for_integration', 'cell_id_original', 'leiden_scvi_0.1', 'leiden_scvi_0.2', 'leiden_scvi_0.3', 'leiden_scvi_0.4', 'leiden_scvi_0.5', 'leiden_scvi_HVG8000_r0_1', 'leiden_scvi_HVG8000_r0_15', 'leiden_scvi_HVG8000_r0_2', 'leid

In [12]:
# =========================
# 9) ATOMIC PUBLICATION + OPTIONAL HARD-LINK ALIAS
# =========================

if FULLGENE_OUTPUT_H5AD.exists():
    FULLGENE_OUTPUT_H5AD.unlink()


os.replace(
    temporary_output,
    FULLGENE_OUTPUT_H5AD,
)


print(
    "[PUBLISHED]"
)

print(
    FULLGENE_OUTPUT_H5AD
)


# Avoid a second 16+ GiB physical copy. Create a filesystem hard link
# only when the current filesystem supports it.
alias_created = False

if FULLGENE_R06_ALIAS_H5AD.exists():
    try:
        if (
            FULLGENE_R06_ALIAS_H5AD.samefile(
                FULLGENE_OUTPUT_H5AD
            )
        ):
            alias_created = True
        else:
            FULLGENE_R06_ALIAS_H5AD.unlink()
    except Exception:
        FULLGENE_R06_ALIAS_H5AD.unlink()


if not alias_created:
    try:
        os.link(
            FULLGENE_OUTPUT_H5AD,
            FULLGENE_R06_ALIAS_H5AD,
        )

        alias_created = True

        print(
            "[HARD-LINK ALIAS CREATED]"
        )

        print(
            FULLGENE_R06_ALIAS_H5AD
        )

    except OSError as error:
        print(
            "[INFO] Filesystem hard-link alias was not created:"
        )

        print(
            repr(
                error
            )
        )

        print(
            "This is non-fatal. The canonical downstream file is:"
        )

        print(
            FULLGENE_OUTPUT_H5AD
        )

[PUBLISHED]
G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters.h5ad
[HARD-LINK ALIAS CREATED]
G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters_r0_6.h5ad


In [13]:
# =========================
# 10) REPORTS
# =========================

cluster_sizes = (
    final_obs[
        MAIN_CLUSTER_KEY
    ]
    .astype(str)
    .value_counts()
    .rename_axis(
        "cluster"
    )
    .reset_index(
        name="n_cells"
    )
)

cluster_sizes[
    "cluster_numeric_sort"
] = (
    cluster_sizes[
        "cluster"
    ]
    .map(
        lambda x:
            int(x)
            if str(x).isdigit()
            else 999999
    )
)

cluster_sizes = (
    cluster_sizes
    .sort_values(
        "cluster_numeric_sort"
    )
    .drop(
        columns=[
            "cluster_numeric_sort"
        ]
    )
)

cluster_sizes[
    "cluster_key"
] = MAIN_CLUSTER_KEY


write_tsv_replace(
    cluster_sizes,
    MAIN_CLUSTER_SIZES_TSV,
)

display(
    cluster_sizes
)


composition_rows = []

for cluster, sub in final_obs.groupby(
    MAIN_CLUSTER_KEY,
    observed=True,
):

    row = {
        "cluster":
            str(
                cluster
            ),

        "cluster_key":
            MAIN_CLUSTER_KEY,

        "n_cells":
            int(
                sub.shape[0]
            ),
    }


    for column in (
        "core_dataset",
        "condition",
        "patient_id",
        "sample_id",
    ):

        if column not in sub.columns:
            continue


        counts = (
            sub[column]
            .astype(str)
            .value_counts(
                dropna=False
            )
        )


        row[
            f"top_{column}"
        ] = (
            str(
                counts.index[0]
            )
            if len(counts)
            else "NA"
        )

        row[
            f"top_{column}_n"
        ] = (
            int(
                counts.iloc[0]
            )
            if len(counts)
            else 0
        )

        row[
            f"top_{column}_fraction"
        ] = (
            float(
                counts.iloc[0]
                / sub.shape[0]
            )
            if (
                len(counts)
                and sub.shape[0] > 0
            )
            else np.nan
        )

        row[
            f"n_{column}_present"
        ] = int(
            counts.shape[0]
        )


    composition_rows.append(
        row
    )


composition = pd.DataFrame(
    composition_rows
)


write_tsv_replace(
    composition,
    MAIN_CLUSTER_COMPOSITION_TSV,
)


cluster_summary_rows = []

cluster_keys = [
    column
    for column
    in final_obs.columns
    if (
        column.startswith(
            "leiden_scvi_HVG8000_"
        )
        or column
        in (
            "leiden_scvi_main",
            "leiden_scvi_HVG8000_main",
            "leiden_scvi_HVG8000_refinement",
        )
    )
]


for key in cluster_keys:

    counts = (
        final_obs[key]
        .astype(str)
        .value_counts()
    )

    cluster_summary_rows.append(
        {
            "cluster_key":
                key,

            "n_clusters":
                int(
                    counts.shape[0]
                ),

            "min_cluster_size":
                int(
                    counts.min()
                ),

            "max_cluster_size":
                int(
                    counts.max()
                ),

            "median_cluster_size":
                float(
                    counts.median()
                ),

            "is_main":
                key
                in (
                    MAIN_CLUSTER_KEY,
                    "leiden_scvi_main",
                    "leiden_scvi_HVG8000_main",
                ),

            "is_refinement":
                key
                in (
                    REFINEMENT_CLUSTER_KEY,
                    "leiden_scvi_HVG8000_refinement",
                ),
        }
    )


cluster_key_summary = pd.DataFrame(
    cluster_summary_rows
)


write_tsv_replace(
    cluster_key_summary,
    CLUSTER_KEY_SUMMARY_TSV,
)


report = pd.DataFrame(
    [
        {
            "fullgene_output_h5ad":
                str(
                    FULLGENE_OUTPUT_H5AD
                ),

            "r0_6_hardlink_alias":
                str(
                    FULLGENE_R06_ALIAS_H5AD
                ),

            "hardlink_alias_created":
                alias_created,

            "canonical_reference_h5ad":
                str(
                    CANONICAL_REFERENCE_H5AD
                ),

            "n_cells":
                EXPECTED_N_CELLS,

            "n_genes":
                EXPECTED_FULLGENE_GENES,

            "main_cluster_key":
                MAIN_CLUSTER_KEY,

            "main_n_clusters":
                HISTORICAL_R06_N_CLUSTERS,

            "X_scVI_shape":
                str(
                    EXPECTED_X_SCVI_SHAPE
                ),

            "X_umap_shape":
                str(
                    EXPECTED_X_UMAP_SHAPE
                ),

            "current_standardized_metadata_preserved":
                True,

            "historical_cluster_assignments_preserved":
                True,

            "fresh_rerun_used_for_article_clusters":
                False,

            "full_expression_matrix_loaded_into_ram":
                False,
        }
    ]
)


write_tsv_replace(
    report,
    TRANSFER_REPORT_TSV,
)


manifest = pd.DataFrame(
    [
        {
            "name":
                "fullgene_counts_input",

            "path":
                str(
                    FULLGENE_COUNTS_H5AD
                ),

            "exists":
                FULLGENE_COUNTS_H5AD.exists(),
        },

        {
            "name":
                "standardized_metadata_input",

            "path":
                str(
                    STANDARDIZED_METADATA_H5AD
                ),

            "exists":
                STANDARDIZED_METADATA_H5AD.exists(),
        },

        {
            "name":
                "canonical_historical_reference",

            "path":
                str(
                    CANONICAL_REFERENCE_H5AD
                ),

            "exists":
                CANONICAL_REFERENCE_H5AD.exists(),
        },

        {
            "name":
                "canonical_fullgene_output",

            "path":
                str(
                    FULLGENE_OUTPUT_H5AD
                ),

            "exists":
                FULLGENE_OUTPUT_H5AD.exists(),
        },

        {
            "name":
                "r0_6_hardlink_alias",

            "path":
                str(
                    FULLGENE_R06_ALIAS_H5AD
                ),

            "exists":
                FULLGENE_R06_ALIAS_H5AD.exists(),
        },
    ]
)


write_tsv_replace(
    manifest,
    TRANSFER_MANIFEST_TSV,
)


display(
    report
)

display(
    manifest
)

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\fullgene_canonical_HVG8000_r0_6_cluster_sizes.tsv shape=(18, 3)


,cluster,n_cells,cluster_key
0,0,61646,leiden_scvi_HVG8000_r0_6
1,1,59791,leiden_scvi_HVG8000_r0_6
2,2,38634,leiden_scvi_HVG8000_r0_6
3,3,35355,leiden_scvi_HVG8000_r0_6
4,4,32896,leiden_scvi_HVG8000_r0_6
5,5,23977,leiden_scvi_HVG8000_r0_6
6,6,22943,leiden_scvi_HVG8000_r0_6
7,7,22525,leiden_scvi_HVG8000_r0_6
8,8,14809,leiden_scvi_HVG8000_r0_6
9,9,13864,leiden_scvi_HVG8000_r0_6


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\fullgene_canonical_HVG8000_r0_6_cluster_composition.tsv shape=(18, 19)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\fullgene_canonical_HVG8000_cluster_key_summary.tsv shape=(19, 7)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\transfer_canonical_HVG8000_scVI_to_fullgene_report.tsv shape=(1, 14)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\transfer_canonical_HVG8000_scVI_to_fullgene_manifest.tsv shape=(5, 3)


,fullgene_output_h5ad,r0_6_hardlink_alias,hardlink_alias_created,canonical_reference_h5ad,n_cells,n_genes,main_cluster_key,main_n_clusters,X_scVI_shape,X_umap_shape,current_standardized_metadata_preserved,historical_cluster_assignments_preserved,fresh_rerun_used_for_article_clusters,full_expression_matrix_loaded_into_ram
0,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True,G:\Repository\glioma-cnv-single-cell-atlas\dat...,351427,15176,leiden_scvi_HVG8000_r0_6,18,"(351427, 30)","(351427, 2)",True,True,False,False


,name,path,exists
0,fullgene_counts_input,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
1,standardized_metadata_input,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
2,canonical_historical_reference,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
3,canonical_fullgene_output,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
4,r0_6_hardlink_alias,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True


In [14]:
# =========================
# 11) FINAL BACKED VALIDATION
# =========================

final_check = sc.read_h5ad(
    FULLGENE_OUTPUT_H5AD,
    backed="r",
)

try:
    final_counts = (
        final_check.obs[
            MAIN_CLUSTER_KEY
        ]
        .astype(str)
        .value_counts()
    )

    print(
        "Saved and validated:",
        FULLGENE_OUTPUT_H5AD,
    )

    print(
        "Validated shape:",
        final_check.shape,
    )

    print(
        "Validated counts layer:",
        "counts"
        in final_check.layers,
    )

    print(
        "Validated X_scVI:",
        final_check.obsm[
            "X_scVI"
        ].shape,
    )

    print(
        "Validated X_umap:",
        final_check.obsm[
            "X_umap"
        ].shape,
    )

    print(
        "Validated canonical r=0.60 clusters:",
        final_counts.shape[0],
    )

    print(
        "Validated canonical cluster min/max/median:",
        int(
            final_counts.min()
        ),
        int(
            final_counts.max()
        ),
        float(
            final_counts.median()
        ),
    )

    print(
        "Validated core datasets:",
        sorted(
            final_check.obs[
                "core_dataset"
            ]
            .astype(str)
            .unique()
            .tolist()
        ),
    )

finally:
    final_check.file.close()

Saved and validated: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters.h5ad
Validated shape: (351427, 15176)
Validated counts layer: True
Validated X_scVI: (351427, 30)
Validated X_umap: (351427, 2)
Validated canonical r=0.60 clusters: 18
Validated canonical cluster min/max/median: 105 61646 14336.5
Validated core datasets: ['DS1_GSE103224', 'DS2_GSE141383', 'DS3_GSE182109', 'DS4_GSE278450']


## Expected successful completion

```text
Saved and validated:
...GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters.h5ad

Validated shape: (351427, 15176)
Validated counts layer: True
Validated X_scVI: (351427, 30)
Validated X_umap: (351427, 2)
Validated canonical r=0.60 clusters: 18
Validated canonical cluster min/max/median:
105 61646 14336.5

Validated core datasets:
['DS1_GSE103224', 'DS2_GSE141383', 'DS3_GSE182109', 'DS4_GSE278450']
```

The large full-gene count matrix is copied sequentially and never materialized in
RAM. The canonical historical reference is lightweight and contains no
expression matrix.

If the filesystem supports hard links, the `_r0_6.h5ad` compatibility path
points to the same physical file and does not consume another ~16 GiB.